# SPINE-GPE v7 — Fechamento da Fase 0 v1.0.0

Fluxo sequencial e fail-closed:

1. **RAIS Substantive Adjudication Lock**;
2. **Decodificação editorial dos perfis RAIS**;
3. **Master Harmonization & Evidence Lock**.

O notebook não relê a RAIS bruta. Ele usa os locks e o Parquet certificado dos vínculos ativos.

In [1]:
from google.colab import drive, files
from pathlib import Path
import json
import shutil
import subprocess
import sys
import zipfile
import pandas as pd

drive.mount('/content/drive', force_remount=False)

ROOT = Path('/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7')
ROOT.mkdir(parents=True, exist_ok=True)

ZIP_NAME = 'SPINE_GPEv7_PHASE0_CLOSURE_PACKAGE_v1.0.0.zip'
ZIP_PATH = Path('/content') / ZIP_NAME

if not ZIP_PATH.is_file():
    print(f'Envie agora o arquivo {ZIP_NAME}.')
    uploaded = files.upload()
    if ZIP_NAME not in uploaded:
        raise FileNotFoundError(f'Arquivo esperado: {ZIP_NAME}')

print('Pacote:', ZIP_PATH)
print('Tamanho MB:', round(ZIP_PATH.stat().st_size / 1024**2, 2))

Mounted at /content/drive
Envie agora o arquivo SPINE_GPEv7_PHASE0_CLOSURE_PACKAGE_v1.0.0.zip.


Saving SPINE_GPEv7_PHASE0_CLOSURE_PACKAGE_v1.0.0.zip to SPINE_GPEv7_PHASE0_CLOSURE_PACKAGE_v1.0.0.zip
Pacote: /content/SPINE_GPEv7_PHASE0_CLOSURE_PACKAGE_v1.0.0.zip
Tamanho MB: 0.03


In [2]:
INSTALL_DIR = ROOT / 'scripts' / 'phase0_closure_v100'
if INSTALL_DIR.exists():
    shutil.rmtree(INSTALL_DIR)
INSTALL_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH) as archive:
    for member in [name for name in archive.namelist() if not name.endswith('/')]:
        relative = Path(member)
        parts = relative.parts[1:] if len(relative.parts) > 1 else relative.parts
        if not parts:
            continue
        target = INSTALL_DIR.joinpath(*parts)
        target.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(member) as src, target.open('wb') as dst:
            shutil.copyfileobj(src, dst)

ENGINE = INSTALL_DIR / 'SPINE_GPEv7_PHASE0_CLOSURE_v1.0.0.py'
CODEBOOK = INSTALL_DIR / 'rais_editorial_codebook_ptbr_v1.0.0.csv'
REQ = INSTALL_DIR / 'requirements_SPINE_GPEv7_PHASE0_CLOSURE_v1.0.0.txt'

assert ENGINE.is_file(), ENGINE
assert CODEBOOK.is_file(), CODEBOOK
assert REQ.is_file(), REQ
print('Instalado em:', INSTALL_DIR)

Instalado em: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/phase0_closure_v100


In [3]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQ)],
    check=True,
)
print('Dependências instaladas.')

Dependências instaladas.


## 1. Audit prévio

O audit confirma a presença dos locks upstream, do Parquet RAIS e do codebook editorial. Não cria os locks finais.

In [4]:
AUDIT_RUN_ID = 'phase0_closure_audit_v100'
cmd_audit = [
    sys.executable, str(ENGINE),
    '--root', str(ROOT),
    '--mode', 'audit',
    '--stage', 'all',
    '--run-id', AUDIT_RUN_ID,
    '--codebook', str(CODEBOOK),
    '--strict',
]

print('Executando audit:')
print(' '.join(cmd_audit))
print()
process = subprocess.Popen(
    cmd_audit,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end='')
audit_exit_code = process.wait()
print()
print('Audit exit code:', audit_exit_code)

Executando audit:
/usr/bin/python3 /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/phase0_closure_v100/SPINE_GPEv7_PHASE0_CLOSURE_v1.0.0.py --root /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7 --mode audit --stage all --run-id phase0_closure_audit_v100 --codebook /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/phase0_closure_v100/rais_editorial_codebook_ptbr_v1.0.0.csv --strict

2026-07-26 16:42:23,213 | INFO | SPINE-GPE Phase 0 Closure v1.0.0 | mode=audit | stage=all | root=/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7
2026-07-26 16:42:26,211 | INFO | Phase 0 closure audit | status=AUDIT_PASSED
{
  "run_id": "phase0_closure_audit_v100",
  "script_version": "1.0.0",
  "schema_version": "spine-gpe-v7-phase0-closure-1.0.0",
  "mode": "audit",
  "status": "AUDIT_PASSED",
  "critical_failures": [],
  "warnings": [],
  "root": "/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7",
  "rais_active_rows": 144800,
  "artifacts": {
    "gates": "/con

In [5]:
AUDIT_LOCK = ROOT / '00_admin' / 'PHASE0_CLOSURE_AUDIT_LOCK.json'
assert AUDIT_LOCK.is_file(), AUDIT_LOCK
audit_lock = json.loads(AUDIT_LOCK.read_text(encoding='utf-8'))
print(json.dumps(audit_lock, ensure_ascii=False, indent=2))
assert audit_exit_code == 0
assert audit_lock['status'] == 'AUDIT_PASSED'
assert audit_lock['critical_failures'] == []
print()
print('PHASE 0 CLOSURE AUDIT PASSED')

{
  "run_id": "phase0_closure_audit_v100",
  "script_version": "1.0.0",
  "schema_version": "spine-gpe-v7-phase0-closure-1.0.0",
  "mode": "audit",
  "status": "AUDIT_PASSED",
  "critical_failures": [],
  "warnings": [],
  "root": "/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7",
  "rais_active_rows": 144800,
  "artifacts": {
    "gates": "/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/05_outputs/tables/phase0_closure/phase0_closure_audit_gates_phase0_closure_audit_v100.csv",
    "report": "/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/06_reports/phase0_closure/phase0_closure_audit_report_phase0_closure_audit_v100.md"
  },
  "artifact_hashes": {
    "gates": "e916298bab95ff89782847b46cd1c41f01e28bebafe5346c9d2edc747231eb11",
    "report": "455d418a8e2d166831cb76ec36c6894ffbe914bf4061759b842ff89ca42e128a"
  },
  "created_at_utc": "2026-07-26T16:42:26.095032+00:00"
}

PHASE 0 CLOSURE AUDIT PASSED


## 2. Execução completa

Esta célula produz os três locks, seus freezes e as tabelas finais da Fase 0.

In [6]:
FULL_RUN_ID = 'phase0_closure_final_v100'
cmd_full = [
    sys.executable, str(ENGINE),
    '--root', str(ROOT),
    '--mode', 'full',
    '--stage', 'all',
    '--run-id', FULL_RUN_ID,
    '--codebook', str(CODEBOOK),
    '--strict',
]

print('Executando full:')
print(' '.join(cmd_full))
print()
process = subprocess.Popen(
    cmd_full,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end='')
full_exit_code = process.wait()
print()
print('Full exit code:', full_exit_code)

Executando full:
/usr/bin/python3 /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/phase0_closure_v100/SPINE_GPEv7_PHASE0_CLOSURE_v1.0.0.py --root /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7 --mode full --stage all --run-id phase0_closure_final_v100 --codebook /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/phase0_closure_v100/rais_editorial_codebook_ptbr_v1.0.0.csv --strict

2026-07-26 16:43:53,836 | INFO | SPINE-GPE Phase 0 Closure v1.0.0 | mode=full | stage=all | root=/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7
2026-07-26 16:43:55,140 | INFO | RAIS substantive adjudication | status=ADJUDICATED
/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/phase0_closure_v100/SPINE_GPEv7_PHASE0_CLOSURE_v1.0.0.py:611: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old

In [9]:
import subprocess
import sys
from pathlib import Path

ROOT = Path(
    "/content/drive/MyDrive/"
    "aCidadeAlgoritmica/SPINE-GPEv7"
)

INSTALL_DIR = (
    ROOT
    / "scripts"
    / "phase0_closure_v101"
)

ENGINE = (
    INSTALL_DIR
    / "SPINE_GPEv7_PHASE0_CLOSURE_v1.0.1.py"
)

CODEBOOK = (
    INSTALL_DIR
    / "rais_editorial_codebook_ptbr_v1.0.1.csv"
)

RUN_ID = "phase0_closure_final_v101"

cmd = [
    sys.executable,
    str(ENGINE),
    "--root",
    str(ROOT),
    "--mode",
    "full",
    "--stage",
    "all",
    "--run-id",
    RUN_ID,
    "--codebook",
    str(CODEBOOK),
    "--strict",
]

print(" ".join(cmd))

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

assert process.stdout is not None

for line in process.stdout:
    print(line, end="")

exit_code = process.wait()

print("\nExit code:", exit_code)

/usr/bin/python3 /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/phase0_closure_v101/SPINE_GPEv7_PHASE0_CLOSURE_v1.0.1.py --root /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7 --mode full --stage all --run-id phase0_closure_final_v101 --codebook /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/phase0_closure_v101/rais_editorial_codebook_ptbr_v1.0.1.csv --strict
2026-07-26 17:03:28,299 | INFO | SPINE-GPE Phase 0 Closure v1.0.1 | mode=full | stage=all | root=/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7
2026-07-26 17:03:30,179 | INFO | RAIS substantive adjudication | status=ADJUDICATED
2026-07-26 17:03:41,369 | INFO | RAIS editorial profile | status=EDITORIAL_CERTIFIED
2026-07-26 17:03:43,543 | INFO | Phase 0 master lock | status=PHASE0_CERTIFIED
{
  "rais_adjudication": {
    "run_id": "phase0_closure_final_v101",
    "script_version": "1.0.1",
    "schema_version": "spine-gpe-v7-phase0-closure-1.0.1",
    "component": "RAIS_SUBSTANTIVE_ADJUDICA

In [10]:
from pathlib import Path
import hashlib
import json

ROOT = Path(
    "/content/drive/MyDrive/"
    "aCidadeAlgoritmica/SPINE-GPEv7"
)

MASTER_LOCK = (
    ROOT
    / "00_admin"
    / "SPINE_GPE_PHASE0_MASTER_LOCK.json"
)

MASTER_FREEZE = (
    ROOT
    / "00_admin"
    / "SPINE_GPE_PHASE0_MASTER_FREEZE.json"
)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


assert MASTER_LOCK.is_file(), MASTER_LOCK
assert MASTER_FREEZE.is_file(), MASTER_FREEZE

lock = json.loads(
    MASTER_LOCK.read_text(encoding="utf-8")
)

freeze = json.loads(
    MASTER_FREEZE.read_text(encoding="utf-8")
)

print("Master status:", lock.get("status"))
print("Freeze status:", freeze.get("status"))
print("Next phase:", lock.get("next_phase"))
print("Master SHA-256:", sha256_file(MASTER_LOCK))
print("Freeze SHA-256:", sha256_file(MASTER_FREEZE))

assert lock["status"] == "PHASE0_CERTIFIED"
assert lock["critical_failures"] == []
assert freeze["status"] == "FROZEN"

print("\nPHASE 0 CERTIFIED AND FROZEN")

Master status: PHASE0_CERTIFIED
Freeze status: FROZEN
Next phase: FASE_1_EVIDENCE_FOUNDATION
Master SHA-256: 38d3e5e5380032fd61997a8e913113d5430f8670af9c51870a47edcc97232c53
Freeze SHA-256: 3ee7e7eca7f71d6e4216621e30bb71541829fa5709be13d2a4ffb46155312454

PHASE 0 CERTIFIED AND FROZEN


In [7]:
ADMIN = ROOT / '00_admin'
LOCKS = {
    'rais_adjudication': ADMIN / 'RAIS_SUBSTANTIVE_ADJUDICATION_LOCK.json',
    'rais_editorial': ADMIN / 'RAIS_EDITORIAL_PROFILE_LOCK.json',
    'phase0_master': ADMIN / 'SPINE_GPE_PHASE0_MASTER_LOCK.json',
}

payloads = {}
for name, path in LOCKS.items():
    assert path.is_file(), path
    payloads[name] = json.loads(path.read_text(encoding='utf-8'))
    print(name, '=>', payloads[name].get('status'))

assert full_exit_code == 0
assert payloads['rais_adjudication']['status'] == 'ADJUDICATED'
assert payloads['rais_editorial']['status'] == 'EDITORIAL_CERTIFIED'
assert payloads['phase0_master']['status'] == 'PHASE0_CERTIFIED'
assert payloads['phase0_master']['critical_failures'] == []

print()
print('FASE 0 CERTIFICADA E CONGELADA')

rais_adjudication => ADJUDICATED
rais_editorial => EDITORIAL_BLOCKED


AssertionError: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/00_admin/SPINE_GPE_PHASE0_MASTER_LOCK.json

## 3. Revisão dos outputs finais

In [ ]:
master = payloads['phase0_master']
rais_adj = payloads['rais_adjudication']
rais_ed = payloads['rais_editorial']

income = pd.read_csv(rais_adj['artifacts']['income_adjudication'])
editorial = pd.read_csv(rais_ed['artifacts']['editorial_profile'])
evidence = pd.read_csv(master['artifacts']['evidence_matrix'])
rules = pd.read_csv(master['artifacts']['comparison_rules'])
claims = pd.read_csv(master['artifacts']['claim_registry'])

print('RAIS — adjudicação remuneratória')
display(income)

print('RAIS — perfil editorial')
display(editorial.head(200))

print('Matriz de evidência')
display(evidence)

print('Regras de comparação')
display(rules)

print('Claim registry')
display(claims)

In [ ]:
MASTER_FREEZE = ADMIN / 'SPINE_GPE_PHASE0_MASTER_FREEZE.json'
assert MASTER_FREEZE.is_file(), MASTER_FREEZE
freeze = json.loads(MASTER_FREEZE.read_text(encoding='utf-8'))
print(json.dumps(freeze, ensure_ascii=False, indent=2))
assert freeze['status'] == 'FROZEN'
assert freeze['read_only'] is True
print()
print('PRÓXIMA FASE:', freeze['next_phase'])